In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os, getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

/var/folders/j8/5slw2pqj0dz3mknyjkc_37500000gn/T/ipykernel_95557/256503748.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [4]:
loader = PyPDFLoader("medical_report.pdf")
docs = loader.load()

In [5]:
len(docs)

9

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)

In [7]:
len(splitted_docs)

26

In [8]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)


In [9]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf
        documents have details about medical reports.
    """

    print("Tool Called: ", query)
    docs = vector_store.similarity_search(query=query, k = 4)
    context = ""

    for doc in docs:
        context = doc.page_content + "\n\n"
    
    return context

In [10]:
llm = ChatOpenAI(model="gpt-5")

In [11]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [12]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [13]:
query = "What is the name of patient, and what is the name of Doctors"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

Tool Called:  physician name
Tool Called:  doctor name
Tool Called:  Patient Name:
Tool Called:  Doctor:
Tool Called:  patient name


In [14]:
result = response["messages"][-1].content

In [15]:
print(result)

- Patient name: Ms. Nikita Chudhary
- Doctors:
  - Dr. Nitin Nahar (Referring doctor)
  - Dr. Swapnil Gupta
  - Dr. Kiran Bhargava Pathak
  - Dr. Beena Chandrasekhar
  - Dr. Nagarjun Sai Jaine
  - Dr. Sunanda
